# Selective RAD kernels and conditioning

The five coefficient cases follow Araya et al. (2024), section 5.3. The recorded spectra use physical nodal and constant bases before solver equilibration. Published marker uncertainty, geometric choices and any uncertified solves remain explicit.
The default renders the retained scalar spectra and figures. Complete current
spectral acquisition is available through `python -m scripts.run_notebooks
45 --study`. Original independent native field comparisons are a separate
historical replay and require their attributed coefficient archives.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/252ace00467e5b04079ab20baab1dad486caa7df0a5367cd640727304bf12645/45_rad_conditioning-companion.zip"
COMPANION_SHA256 = "252ace00467e5b04079ab20baab1dad486caa7df0a5367cd640727304bf12645"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("transport/45_rad_conditioning.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


import os

HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"


In [ ]:
from pathlib import Path
import json
from IPython.display import Image, display
record = json.loads((root / "examples/results/rad-conditioning.json").read_text())
rows = record["records"]
assert set(r["case"] for r in rows) == {1, 2, 3, 4, 5}
[(s, sum(r["study"] == s for r in rows)) for s in ("epsilon", "omega", "local", "skeleton", "quadrature")]

In [ ]:
[(r["case"], r["kernel_cells"], r["global_condition"]) for r in rows
 if r["study"] == "epsilon" and r["epsilon"] == 1.0]

The local ordinates differ from the published values. A matrix condition number does not certify a reconstructed solution: records that fail the unchanged residual criterion retain `not_certified` status.

In [ ]:
[(r["study"], r["case"], r["epsilon"], r["omega"]) for r in rows if r["status"] != "verified"]

In [ ]:
for name in ("epsilon", "omega", "mesh"):
    display(Image(filename=str(root / f"docs/figures/rad-conditioning/{name}.png")))

Independent DOLFINx/UFL complete-system comparisons use the same material, boundary conditions and local/skeletal spaces. Saved coordinates, connectivity and polynomial ordering permit field replay. The native code does not reuse PyMHM local matrices or condensation. Historical connectivity remains unspecified.
The saved original coefficient archives are not distributed. Their checksum
checks below run only with `--historical`; rendering scalar spectra does not
execute the independent reference solver.


In [ ]:
if HISTORICAL_REPLAY:
    import hashlib
    native = json.loads((root / "examples/results/rad-native/verification.json").read_text())
    assert native["accepted"]
    native_rows = [row for row in native["rows"] if row["kind"] == "conditioning"]
    assert len(native_rows) == 10
    for row in native_rows:
        archive = root / "examples/results/rad-native" / row["archive"]
        assert hashlib.sha256(archive.read_bytes()).hexdigest() == row["archive_sha256"]
        assert row["accepted"]
    [(row["n"], row["local_refinement"], row["case"], row["epsilon"],
      row["physical_norms"]["20"]["physical_flux"]["relative_difference"])
     for row in native_rows]
else:
    print("Original native field checks require --historical and the attributed archives.")
